In [2]:
!pip install rdkit pandas scikit-learn

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem
print("RDKit works")

RDKit works


In [4]:
import pandas as pd
df = pd.read_csv("https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv")
print(df.shape)
print(df.head())
print(df['p_np'].value_counts())

(2050, 4)
   num                  name  p_np  \
0    1            Propanolol     1   
1    2  Terbutylchlorambucil     1   
2    3                 40730     1   
3    4                    24     1   
4    5           cloxacillin     1   

                                              smiles  
0                   [Cl].CC(C)NCC(O)COc1cccc2ccccc12  
1           C(=O)(OC(C)(C)C)CCCc1ccc(cc1)N(CCCl)CCCl  
2  c12c3c(N4CCN(C)CC4)c(F)cc1c(c(C(O)=O)cn2C(C)CO...  
3                   C1CCN(CC1)Cc1cccc(c1)OCCCNC(=O)C  
4  Cc1onc(c2ccccc2Cl)c1C(=O)N[C@H]3[C@H]4SC(C)(C)...  
p_np
1    1567
0     483
Name: count, dtype: int64


In [5]:
mol = Chem.MolFromSmiles(df['smiles'][0])
fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
print(len(fp))

1024


[09:29:37] DEPRECATION WARNING: please use MorganGenerator


In [6]:
df['mol'] = df['smiles'].apply(Chem.MolFromSmiles)
df = df[df['mol'].notna()]
print(df.shape)

[09:29:56] Explicit valence for atom # 1 N, 4, is greater than permitted
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] Explicit valence for atom # 6 N, 4, is greater than permitted
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] Explicit valence for atom # 6 N, 4, is greater than permitted
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] Explicit valence for atom # 11 N, 4, is greater than pe

(2039, 5)


[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors
[09:29:56] WARNING: not removing hydrogen atom without neighbors


In [7]:
import numpy as np
def get_fp(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
    return np.array(fp)

df['fp'] = df['smiles'].apply(get_fp)
df = df[df['fp'].notna()]
X = np.stack(df['fp'].values)
y = df['p_np'].values
print(X.shape, y.shape)

[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerator
[09:31:09] DEPRECATION WARNING: please use MorganGenerat

(2039, 1024) (2039,)


[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerator
[09:31:11] DEPRECATION WARNING: please use MorganGenerat

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(1631, 1024) (408, 1024)
(1631,) (408,)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.8921568627450981
AUC: 0.9285022702991452
              precision    recall  f1-score   support

           0       0.92      0.59      0.72        96
           1       0.89      0.98      0.93       312

    accuracy                           0.89       408
   macro avg       0.90      0.79      0.83       408
weighted avg       0.89      0.89      0.88       408



In [10]:
from sklearn.svm import SVC
svm = SVC(C=1.0, kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)
y_prob = svm.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.9093137254901961
AUC: 0.905665731837607
              precision    recall  f1-score   support

           0       0.97      0.64      0.77        96
           1       0.90      0.99      0.94       312

    accuracy                           0.91       408
   macro avg       0.93      0.81      0.86       408
weighted avg       0.91      0.91      0.90       408



In [11]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
y_prob = knn.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.8848039215686274
AUC: 0.8469551282051282
              precision    recall  f1-score   support

           0       0.95      0.54      0.69        96
           1       0.88      0.99      0.93       312

    accuracy                           0.88       408
   macro avg       0.91      0.77      0.81       408
weighted avg       0.89      0.88      0.87       408



In [12]:
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred = mlp.predict(X_test)
y_prob = mlp.predict_proba(X_test)[:, 1]
print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Accuracy: 0.875
AUC: 0.8907084668803419
              precision    recall  f1-score   support

           0       0.83      0.59      0.69        96
           1       0.88      0.96      0.92       312

    accuracy                           0.88       408
   macro avg       0.86      0.78      0.81       408
weighted avg       0.87      0.88      0.87       408



In [13]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    X, y, cv=5, scoring='roc_auc'
)
print(scores)
print("Mean AUC:", scores.mean())

[0.7734375  0.82178152 0.79972957 0.957916   0.98522267]
Mean AUC: 0.8676174510796221


In [14]:
from sklearn.inspection import permutation_importance
result = permutation_importance(
    rf, X_test, y_test, n_repeats=3,
    random_state=42, scoring='roc_auc', n_jobs=-1
)
importances = result.importances_mean
top = importances.argsort()[::-1][:10]
for i in top:
    print(f"Bit {i}: {importances[i]:.4f}")

Bit 357: 0.0076
Bit 926: 0.0045
Bit 233: 0.0045
Bit 725: 0.0042
Bit 935: 0.0030
Bit 23: 0.0028
Bit 754: 0.0026
Bit 893: 0.0021
Bit 803: 0.0020
Bit 904: 0.0016


In [15]:
import pandas as pd
results = pd.DataFrame([
    ["Random Forest", 0.892, 0.929],
    ["SVM", 0.909, 0.906],
    ["kNN", 0.885, 0.847],
    ["Neural network MLP", 0.875, 0.891],
], columns=["Model", "Accuracy", "AUC"])
print(results)
results.to_csv("bbbp_results.csv", index=False)
print("Saved bbbp_results.csv")

                Model  Accuracy    AUC
0       Random Forest     0.892  0.929
1                 SVM     0.909  0.906
2                 kNN     0.885  0.847
3  Neural network MLP     0.875  0.891
Saved bbbp_results.csv
